# Demand Forecasting Pipeline — Prophet Baseline

**Goal:** Predict weekly SKU demand and generate automated reorder suggestions.

**Dataset:** DataCo Supply Chain (180k orders, 118 SKUs, Jan 2015 – Jan 2018)

**Pipeline steps:**
1. Load & aggregate raw orders → weekly SKU demand
2. Train/test split (last 8 weeks held out)
3. Fit Prophet forecast per SKU
4. Evaluate: MAE and MAPE
5. Reorder decision logic → automated purchase suggestions

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

DATA_PATH = 'DataCoSupplyChainDataset.csv'   # adjust path if needed
FORECAST_HORIZON = 8   # weeks ahead to forecast
TEST_WEEKS       = 8   # weeks held out for evaluation
LEAD_TIME_WEEKS  = 2   # supplier lead time assumption
SERVICE_LEVEL    = 1.65  # z-score for ~95% service level

## 1. Load & Aggregate

From 180k order-line rows → weekly units sold per SKU.

We only need three columns:
- `order date` — when the order was placed  
- `Product Card Id` — unique SKU identifier  
- `Order Item Quantity` — units sold per line

In [ ]:
raw = pd.read_csv(DATA_PATH, encoding='latin1',
                  usecols=['order date (DateOrders)',
                           'Product Card Id',
                           'Product Name',
                           'Order Item Quantity'])

raw['order_date'] = pd.to_datetime(raw['order date (DateOrders)'])
raw['week']       = raw['order_date'].dt.to_period('W').dt.start_time

weekly = (
    raw
    .groupby(['Product Card Id', 'Product Name', 'week'],
             as_index=False)['Order Item Quantity']
    .sum()
    .rename(columns={
        'Product Card Id':    'product_id',
        'Product Name':       'product_name',
        'Order Item Quantity':'units_sold'
    })
    .sort_values(['product_id', 'week'])
    .reset_index(drop=True)
)

# Drop the last week — dataset ends mid-week so it's incomplete
last_week = weekly['week'].max()
weekly = weekly[weekly['week'] < last_week].copy()

print(f"Rows after aggregation : {len(weekly):,}")
print(f"Unique SKUs            : {weekly['product_id'].nunique()}")
print(f"Date range             : {weekly['week'].min().date()} → {weekly['week'].max().date()}")

In [ ]:
# Keep only SKUs with full 144-week history — ensures enough data for forecasting
week_counts = weekly.groupby('product_id')['week'].count()
full_skus   = week_counts[week_counts >= 130].index
weekly      = weekly[weekly['product_id'].isin(full_skus)].copy()

sku_lookup = weekly[['product_id','product_name']].drop_duplicates().set_index('product_id')['product_name']

print(f"SKUs with sufficient history: {len(full_skus)}")
print()
for pid in full_skus:
    print(f"  {pid:>6}  {sku_lookup[pid]}")

### Visualise demand for one SKU

In [ ]:
example_id   = full_skus[0]
example_name = sku_lookup[example_id]
example_df   = weekly[weekly['product_id'] == example_id]

fig, ax = plt.subplots()
ax.plot(example_df['week'], example_df['units_sold'], linewidth=1.2, color='#2563eb')
ax.set_title(f'Weekly demand — {example_name}', fontsize=13)
ax.set_xlabel('Week')
ax.set_ylabel('Units sold')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 2. Train / Test Split

Hold out the **last 8 weeks** as the test set.  
The model never sees these rows during training.

In [ ]:
def split_sku(df_sku, test_weeks=TEST_WEEKS):
    """Split a single SKU's time series into train and test."""
    df_sku = df_sku.sort_values('week').reset_index(drop=True)
    return df_sku.iloc[:-test_weeks], df_sku.iloc[-test_weeks:]

# Quick check on one SKU
ex_train, ex_test = split_sku(weekly[weekly['product_id'] == example_id])
print(f"Train: {len(ex_train)} weeks  ({ex_train['week'].min().date()} → {ex_train['week'].max().date()})")
print(f"Test : {len(ex_test)} weeks  ({ex_test['week'].min().date()} → {ex_test['week'].max().date()})")

## 3. Prophet Forecast

Prophet expects a dataframe with two columns: `ds` (date) and `y` (value).  
We enable yearly seasonality to capture annual demand patterns.  
We also add a **lower bound of 0** — demand cannot be negative.

In [ ]:
def fit_prophet(train_df):
    """Fit a Prophet model on a single SKU's training data."""
    df = train_df[['week', 'units_sold']].rename(
        columns={'week': 'ds', 'units_sold': 'y'}
    )
    model = Prophet(
        yearly_seasonality  = True,
        weekly_seasonality  = False,  # weekly data — no intra-week pattern
        daily_seasonality   = False,
        seasonality_mode    = 'multiplicative',  # demand spikes scale with level
        interval_width      = 0.95
    )
    model.fit(df)
    return model


def predict_prophet(model, horizon=FORECAST_HORIZON):
    """Generate forecast for the next `horizon` weeks."""
    future   = model.make_future_dataframe(periods=horizon, freq='W')
    forecast = model.predict(future)
    # Clip negative predictions to 0
    forecast['yhat']       = forecast['yhat'].clip(lower=0)
    forecast['yhat_lower'] = forecast['yhat_lower'].clip(lower=0)
    forecast['yhat_upper'] = forecast['yhat_upper'].clip(lower=0)
    return forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]

In [ ]:
# Run forecast for all SKUs — collect results
results = []

for pid in full_skus:
    sku_df = weekly[weekly['product_id'] == pid].copy()
    train, test = split_sku(sku_df)

    model    = fit_prophet(train)
    forecast = predict_prophet(model, horizon=TEST_WEEKS)

    # Align forecast tail with test period
    pred = forecast.tail(TEST_WEEKS).reset_index(drop=True)
    test = test.reset_index(drop=True)

    results.append({
        'product_id':   pid,
        'product_name': sku_lookup[pid],
        'model':        model,
        'forecast':     forecast,
        'train':        train,
        'test':         test,
        'pred':         pred,
    })

print(f"Trained {len(results)} models.")

## 4. Evaluate

Two metrics:
- **MAE** (Mean Absolute Error) — average units off per week, easy to explain to stakeholders
- **MAPE** (Mean Absolute Percentage Error) — percentage error, comparable across SKUs with different volumes

In [ ]:
def evaluate(test_df, pred_df):
    """Compute MAE and MAPE for one SKU."""
    actual = test_df['units_sold'].values
    pred   = pred_df['yhat'].values
    mae    = np.mean(np.abs(actual - pred))
    # Guard against zero actuals in MAPE
    nonzero = actual != 0
    mape    = np.mean(np.abs((actual[nonzero] - pred[nonzero]) / actual[nonzero])) * 100
    return round(mae, 1), round(mape, 1)


eval_rows = []
for r in results:
    mae, mape = evaluate(r['test'], r['pred'])
    eval_rows.append({
        'product_id':   r['product_id'],
        'product_name': r['product_name'],
        'MAE':          mae,
        'MAPE_%':       mape,
    })

eval_df = pd.DataFrame(eval_rows).sort_values('MAPE_%')

print(eval_df.to_string(index=False))
print()
print(f"Median MAE  : {eval_df['MAE'].median():.1f} units/week")
print(f"Median MAPE : {eval_df['MAPE_%'].median():.1f}%")

In [ ]:
# Visualise forecast vs actual for the best and worst SKU
best_id  = eval_df.iloc[0]['product_id']
worst_id = eval_df.iloc[-1]['product_id']

def plot_forecast(r, ax, title_suffix=''):
    fc = r['forecast']
    ax.fill_between(fc['ds'], fc['yhat_lower'], fc['yhat_upper'],
                    alpha=0.15, color='#2563eb', label='95% interval')
    ax.plot(fc['ds'], fc['yhat'], color='#2563eb', linewidth=1.5, label='Forecast')
    ax.plot(r['train']['week'], r['train']['units_sold'],
            color='#374151', linewidth=1, label='Actual (train)')
    ax.plot(r['test']['week'], r['test']['units_sold'],
            color='#ef4444', linewidth=2, label='Actual (test)')
    ax.axvline(r['test']['week'].min(), color='gray', linestyle='--', linewidth=0.8)
    ax.set_title(f"{r['product_name'][:40]}{title_suffix}", fontsize=11)
    ax.set_ylabel('Units sold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=9)

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
plot_forecast(next(r for r in results if r['product_id'] == best_id),
              axes[0], title_suffix='  [best MAPE]')
plot_forecast(next(r for r in results if r['product_id'] == worst_id),
              axes[1], title_suffix='  [worst MAPE]')
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Reorder Decision Logic

This is where the forecast connects to a real business decision.

**Logic:**
```
reorder_point = demand_during_leadtime + safety_stock

if current_stock < reorder_point:
    order_qty = forecast_next_8_weeks - current_stock - in_transit
```

Safety stock uses the forecast uncertainty (prediction interval width) as a proxy for demand variability — this is a principled approach: wider intervals = more uncertain = more buffer needed.

> In a real system `current_stock` and `in_transit` come from the warehouse database.
> Here we simulate them to demonstrate the full pipeline.

In [ ]:
def reorder_decision(forecast_df, product_name,
                     current_stock, in_transit=0,
                     lead_time_weeks=LEAD_TIME_WEEKS,
                     z=SERVICE_LEVEL):
    """
    Given a Prophet forecast, decide whether to reorder and how much.

    Parameters
    ----------
    forecast_df   : Prophet forecast dataframe (future periods only)
    current_stock : units currently in warehouse
    in_transit    : units already ordered but not yet arrived
    lead_time_weeks : weeks until supplier delivers
    z             : service-level z-score (1.65 = 95%)

    Returns
    -------
    dict with reorder recommendation
    """
    future = forecast_df.tail(FORECAST_HORIZON).copy()

    # Demand during lead time (weeks 1..lead_time)
    demand_lt = future.head(lead_time_weeks)['yhat'].sum()

    # Safety stock: z * std of weekly demand (estimated from interval width)
    weekly_std   = ((future['yhat_upper'] - future['yhat_lower']) / (2 * z)).mean()
    safety_stock = z * weekly_std * np.sqrt(lead_time_weeks)

    reorder_point = demand_lt + safety_stock

    # Total forecasted demand over horizon (what we need to cover)
    total_demand = future['yhat'].sum()
    order_qty    = max(0, round(total_demand - current_stock - in_transit))

    should_reorder = (current_stock + in_transit) < reorder_point

    return {
        'product_name':    product_name,
        'current_stock':   current_stock,
        'reorder_point':   round(reorder_point),
        'safety_stock':    round(safety_stock),
        'forecast_8w':     round(total_demand),
        'suggested_order': order_qty,
        'action':          'REORDER' if should_reorder else 'ok',
    }

In [ ]:
# Simulate current inventory — set to ~3 weeks of average demand
# (in production this comes from the warehouse system)
rng = np.random.default_rng(seed=42)

suggestions = []
for r in results:
    avg_weekly   = r['train']['units_sold'].mean()
    sim_stock    = int(avg_weekly * rng.uniform(1.5, 4.0))  # random stock level
    sim_transit  = int(avg_weekly * rng.uniform(0, 1.0))

    rec = reorder_decision(
        forecast_df   = r['forecast'],
        product_name  = r['product_name'],
        current_stock = sim_stock,
        in_transit    = sim_transit,
    )
    suggestions.append(rec)

suggestions_df = pd.DataFrame(suggestions).sort_values('action', ascending=False)
print(suggestions_df.to_string(index=False))

In [ ]:
# Final output: purchase order list (only SKUs that need reordering)
purchase_orders = suggestions_df[suggestions_df['action'] == 'REORDER'].copy()

print("=" * 60)
print(f"  AUTOMATED REORDER SUGGESTIONS — {pd.Timestamp.today().date()}")
print("=" * 60)

if len(purchase_orders) == 0:
    print("  All SKUs sufficiently stocked. No reorders needed.")
else:
    for _, row in purchase_orders.iterrows():
        print(f"  {row['product_name'][:45]:<45}"
              f"  order: {row['suggested_order']:>5} units"
              f"  (stock: {row['current_stock']}, ROP: {row['reorder_point']})")

print("=" * 60)
print(f"  {len(purchase_orders)} of {len(suggestions_df)} SKUs flagged for reorder")
print("=" * 60)

## Summary

This notebook demonstrates a **complete demand forecasting pipeline**:

| Step | What it does |
|---|---|
| Aggregation | 180k order rows → weekly SKU time series |
| Forecasting | Prophet with yearly seasonality, 95% prediction intervals |
| Evaluation | MAE and MAPE per SKU, train/test split on last 8 weeks |
| Decision logic | Reorder point = lead-time demand + safety stock |
| Output | Automated purchase order suggestions |

**Next step:** Version 2 will replace Prophet with XGBoost, adding discount rate and category as features — allowing the model to learn how promotions affect demand.